# drift-doctor — Python API demo
No CLI needed. Use directly in notebooks or pipelines.

In [1]:
from drift_doctor import snapshot, check_drift, diff_snapshots
import drift_doctor
print('drift-doctor version:', drift_doctor.__version__)

drift-doctor version: 0.4.0


## 1. Save a reference snapshot

In [2]:
profile = snapshot('reference.csv')
print(f"Snapshot saved — {profile['row_count']} rows, {profile['column_count']} columns")
print('Columns:', list(profile['columns'].keys()))

Snapshot saved — 1000 rows, 6 columns
Columns: ['customer_id', 'age', 'spend', 'status', 'country', 'phone']


## 2. Check current data for drift

In [3]:
result = check_drift('current.csv', skip=['customer_id'])

print('Has drift:', result.has_drift)
print('Summary:', result.summary)
print('Row count: ref=%d  cur=%d' % (result.ref_row_count, result.cur_row_count))

Has drift: True
Summary: {'critical': 1, 'warn': 0, 'total': 1}
Row count: ref=1000  cur=1000


## 3. Inspect findings as a DataFrame

In [4]:
import pandas as pd

rows = [
    {'severity': f.severity.value, 'column': f.column, 'metric': f.metric, 'detail': f.detail or f.description}
    for f in result.findings
]
pd.DataFrame(rows)

,severity,column,metric,detail
0,critical,email,js_divergence,JS=0.658


## 4. Critical findings only

In [5]:
for f in result.critical:
    print(f'CRITICAL  {f.column:15s}  {f.metric:15s}  {f.detail or f.description}')

CRITICAL  email            js_divergence    JS=0.658


## 5. Pipeline guard — raises if critical drift found

In [6]:
try:
    result.raise_on_critical()
except RuntimeError as e:
    print('Pipeline blocked:', e)

Pipeline blocked: Critical drift detected in 1 column(s): email


## 6. Compare two snapshots directly (no raw data)

In [7]:
snapshot('current.csv')

diff = diff_snapshots(
    '.driftdoctor/reference_latest.json',
    '.driftdoctor/current_latest.json',
    skip=['customer_id'],
)
print('Diff summary:', diff.summary)

Diff summary: {'critical': 3, 'warn': 2, 'total': 5}
